In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import json
import time
import hashlib
import sqlite3
import zipfile
import shutil
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/Colab_LLMs/"
    "Maia_Lite_0_3_0_Chat/corpus_v8/runs/"
    "acquisition_v8_5_001/pretraining_production_v1_full"
)

PACK_DIR = ROOT / "packed_zip_v1"
DB = ROOT / "recovery_audit_v4/state_operational_v5.sqlite3"
SSD = Path("/content/maia_corpus_audit")
SSD.mkdir(parents=True, exist_ok=True)

BLOCK_BYTES = 2048 * 4
MAX_PACKS = 28       # Primeiro teste: 1. Depois alterar para 28.
MAX_HOURS = 20

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(4 * 1024 * 1024)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def load_catalog():
    con = sqlite3.connect(DB.as_uri() + "?mode=ro",
                          uri=True, timeout=60)
    try:
        rows = con.execute("""
            SELECT source, split, shard_index,
                   path, blocks, sha256
            FROM shard
            ORDER BY source, split, shard_index
        """).fetchall()
    finally:
        con.close()

    assert len(rows) == 60612
    assert sum(r[4] for r in rows) == 3607119
    return rows

def plan_packs(rows):
    packs = []
    group = []
    size = 0
    limit = 1024 ** 3

    for r in rows:
        n = r[4] * BLOCK_BYTES
        if group and size + n > limit:
            packs.append(group)
            group = []
            size = 0
        group.append(r)
        size += n

    if group:
        packs.append(group)

    assert len(packs) == 28
    return packs

def audit_pack(index, group):
    source, split = group[0][:2]
    name = f"pack_{index:04d}_{source}_{split}.zip"

    remote = PACK_DIR / name
    manifest_path = PACK_DIR / (name + ".json")
    local = SSD / name
    partial = SSD / (name + ".partial")

    meta = json.loads(manifest_path.read_text())

    assert meta["status"] == "complete"
    assert meta["zip_name"] == name
    assert meta["count"] == len(group)
    assert meta["first"] == list(group[0][:3])
    assert meta["last"] == list(group[-1][:3])
    assert meta["raw_bytes"] == sum(
        r[4] * BLOCK_BYTES for r in group
    )
    assert remote.stat().st_size == meta["zip_bytes"]

    expected = {
        f"{src}/{sp}/{src}_{sp}_{idx:06d}.bin":
            (blocks, sha.lower())
        for src, sp, idx, path, blocks, sha in group
    }

    try:
        partial.unlink(missing_ok=True)

        print(f"  Copiando {name} para SSD...", flush=True)
        shutil.copyfile(remote, partial)

        assert partial.stat().st_size == meta["zip_bytes"]
        assert sha256_file(partial) == meta["zip_sha256"]

        os.replace(partial, local)

        seen = set()
        verified_blocks = 0

        with zipfile.ZipFile(local, "r") as z:
            infos = z.infolist()
            assert len(infos) == len(expected)

            for j, info in enumerate(infos, 1):
                arc = info.filename

                assert arc in expected, f"Arquivo inesperado: {arc}"
                assert arc not in seen, f"Duplicado: {arc}"
                assert not info.is_dir()

                blocks, expected_sha = expected[arc]
                assert info.file_size == blocks * BLOCK_BYTES
                assert info.compress_type == zipfile.ZIP_STORED

                h = hashlib.sha256()
                total = 0

                with z.open(info) as stream:
                    while True:
                        buf = stream.read(4 * 1024 * 1024)
                        if not buf:
                            break
                        total += len(buf)
                        h.update(buf)

                assert total == blocks * BLOCK_BYTES
                assert h.hexdigest() == expected_sha, (
                    f"SHA256 incorreto: {arc}"
                )

                seen.add(arc)
                verified_blocks += blocks

                if j % 1000 == 0:
                    print(
                        f"    {j}/{len(infos)} shards",
                        flush=True
                    )

        assert seen == set(expected)

        return len(seen), verified_blocks

    finally:
        local.unlink(missing_ok=True)
        partial.unlink(missing_ok=True)

rows = load_catalog()
packs = plan_packs(rows)

print(f"Shards catalogados: {len(rows):,}")
print(f"Blocos catalogados: {sum(r[4] for r in rows):,}")
print(f"Pacotes planejados: {len(packs)}")
print("=" * 65)

start = time.monotonic()
total_shards = 0
total_blocks = 0
completed = 0

for i, group in enumerate(packs[:MAX_PACKS]):
    if time.monotonic() - start >= MAX_HOURS * 3600:
        print("Limite de tempo atingido.")
        break

    print(f"[{i+1}/28] AUDITANDO", flush=True)
    t = time.monotonic()

    n, b = audit_pack(i, group)

    completed += 1
    total_shards += n
    total_blocks += b

    print(
        f"  PASS: {n:,} shards, {b:,} blocos, "
        f"{time.monotonic()-t:.1f}s",
        flush=True
    )

print("=" * 65)
print(f"Pacotes auditados: {completed}/{len(packs)}")
print(f"Shards verificados: {total_shards:,}")
print(f"Blocos verificados: {total_blocks:,}")

if completed == 28:
    assert total_shards == 60612
    assert total_blocks == 3607119
    print("PASS: INTEGRIDADE COMPLETA DO CORPUS V8")
else:
    print("Auditoria parcial concluída.")

Shards catalogados: 60,612
Blocos catalogados: 3,607,119
Pacotes planejados: 28
[1/28] AUDITANDO
  Copiando pack_0000_fineweb2_es_test.zip para SSD...
    1000/5075 shards
    2000/5075 shards
    3000/5075 shards
    4000/5075 shards
    5000/5075 shards
  PASS: 5,075 shards, 130,952 blocos, 13.7s
[2/28] AUDITANDO
  Copiando pack_0001_fineweb2_es_train.zip para SSD...
  PASS: 876 shards, 131,056 blocos, 23.9s
[3/28] AUDITANDO
  Copiando pack_0002_fineweb2_es_train.zip para SSD...
  PASS: 832 shards, 131,017 blocos, 19.9s
[4/28] AUDITANDO
  Copiando pack_0003_fineweb2_es_train.zip para SSD...
  PASS: 839 shards, 130,942 blocos, 19.7s
[5/28] AUDITANDO
  Copiando pack_0004_fineweb2_es_train.zip para SSD...
  PASS: 865 shards, 130,947 blocos, 21.1s
[6/28] AUDITANDO
  Copiando pack_0005_fineweb2_es_train.zip para SSD...
    1000/12226 shards
    2000/12226 shards
    3000/12226 shards
    4000/12226 shards
    5000/12226 shards
    6000/12226 shards
    7000/12226 shards
    8000/12226 sha